# `07_learning_curves.ipynb` — Data Efficiency, Sampling Strategy & Minimum Step Floor (`Q2`)

## 1. Two Methodological Controls for Honest Learning Curves
When practitioners ask *"How many labeled German documents do I need before QLoRA beats 10-shot prompting?"*, two subtle experimental bugs often distort the answer:

1. **The Constant-Epoch Under-Optimization Trap at Small $N$:**
   With an effective batch size of `16`, a subset of $N=100$ documents has only $\lceil 100 / 16 \rceil = 7$ steps per epoch. Training for 3 epochs executes only **21 optimizer updates**—barely finishing learning-rate warmup! In `training.py`, `compute_training_budget()` enforces a **minimum floor of `100` optimizer steps** (`min_optimizer_steps_floor=100`) so small-$N$ models converge properly.
2. **Uniform Random vs. Rare-Category Stratified Sampling:**
   Because rare categories like `Gepäck` (`0.24%` of train) and `Barrierefreiheit` (`0.50%` of train) are so sparse, a uniform random sample of $N=100$ or $N=250$ documents frequently contains **zero** examples of those categories. Our `sample_learning_curve_subset(..., strategy='stratified')` preserves the natural ~17% empty (`relevance=false`) document rate while guaranteeing rare-label inclusion.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
lc = pd.read_csv('results/metrics/learning_curves_q2.csv')
summary_lc = lc.groupby(['train_examples', 'sampling_strategy', 'optimizer_steps', 'effective_epochs'])[['task_c2_micro_f1', 'task_c2_macro_f1', 'gpu_minutes']].mean().reset_index()
display(summary_lc)
display(Image(filename='results/figures/learning_curve_task_c2.png'))

### Key Takeaways
- Enforcing a 100-step floor prevents small subsets ($N=100, 250$) from collapsing due to under-optimization.
- Stratified multilabel sampling boosts **Macro-F1 by +3–4 points** at small $N$ ($100\text{–}500$) by guaranteeing rare-category coverage.
- **Crossover point:** 4-bit QLoRA surpasses 10-shot `P3` prompting (`0.703` Micro-F1) between **$N=250$ and $N=500$ labeled documents**.